In [3]:
import pyspark

### load the data

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("distributed_project").getOrCreate()
sc = spark.sparkContext

nums_rdd = sc.textFile("/Users/tianyiluo/Documents/Distributed Computing/Distributed_computing_group_project/Distributed_computing_group_project/data/processed/merged/num_2020.csv")
pre_rdd  = sc.textFile("/Users/tianyiluo/Documents/Distributed Computing/Distributed_computing_group_project/Distributed_computing_group_project/data/processed/merged/pre_2020.csv")
sub_rdd  = sc.textFile("/Users/tianyiluo/Documents/Distributed Computing/Distributed_computing_group_project/Distributed_computing_group_project/data/processed/merged/sub_2020.csv")
tag_rdd  = sc.textFile("/Users/tianyiluo/Documents/Distributed Computing/Distributed_computing_group_project/Distributed_computing_group_project/data/processed/merged/tag_2020.csv")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/22 14:46:51 WARN Utils: Your hostname, Tianyis-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.24 instead (on interface en0)
25/11/22 14:46:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/22 14:46:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [12]:
nums_rdd.take(5)

['adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote,quarter,year',
 '0001564590-20-010652,AccountsPayableCurrentAndNoncurrent,us-gaap/2019,20181231,0,USD,,,607000.0,,q1,2020',
 '0000753308-20-000021,LongTermDebtCurrent,us-gaap/2019,20181231,0,USD,LegalEntity=NexteraEnergyResources;,NexteraEnergyResources,602000000.0,,q1,2020',
 '0001393883-20-000011,RevenueFromContractWithCustomerExcludingAssessedTax,us-gaap/2019,20181231,4,USD,BusinessSegments=Other;,,9312000.0,,q1,2020',
 '0001507385-20-000034,StockRedeemedOrCalledDuringPeriodValue,us-gaap/2019,20191231,4,USD,LegalEntity=VEREITOperatingPartnershipL.P.;PartnerCapitalComponents=PreferredStock;PartnerTypeOfPartnersCapitalAccount=GeneralPartner;,,182347000.0,,q1,2020']

In [13]:
pre_rdd.take(5)

['adsh,report,line,stmt,inpth,rfile,tag,version,plabel,negating,quarter,year',
 '0000002178-20-000013,2,3,BS,0,H,CashAndCashEquivalentsAtCarryingValue,us-gaap/2019,Cash and cash equivalents,0,q1,2020',
 '0000002178-20-000013,2,4,BS,0,H,RestrictedCashCurrent,us-gaap/2019,Restricted cash,0,q1,2020',
 '0000002178-20-000013,2,5,BS,0,H,AccountsReceivableNetCurrent,us-gaap/2019,"Accounts receivable, net of allowance for doubtful accounts of $141 and $153, respectively",0,q1,2020',
 '0000002178-20-000013,2,6,BS,0,H,AccountsReceivableRelatedPartiesCurrent,us-gaap/2019,Accounts receivable  related party,0,q1,2020']

In [14]:
sub_rdd.take(5)

['adsh,cik,name,sic,countryba,stprba,cityba,zipba,bas1,bas2,baph,countryma,stprma,cityma,zipma,mas1,mas2,countryinc,stprinc,ein,former,changed,afs,wksi,fye,form,period,fy,fp,filed,accepted,prevrpt,detail,instance,nciks,aciks,quarter,year',
 '0000002178-20-000013,2178,"ADAMS RESOURCES & ENERGY, INC.",5172.0,US,TX,HOUSTON,77027,17 S. BRIAR HOLLOW LN.,,713-881-3600,US,TX,HOUSTON,77001,P O BOX 844,,US,DE,741753147.0,ADAMS RESOURCES & ENERGY INC,19920703.0,2-ACC,0,1231.0,10-K,20191231,2019.0,FY,20200306,2020-03-06 16:50:00.0,0,1,ae-20191231_htm.xml,1,,q1,2020',
 '0000002488-20-000008,2488,ADVANCED MICRO DEVICES INC,3674.0,US,CA,SANTA CLARA,95054,2485 AUGUSTINE DRIVE,,(408) 749-4000,US,CA,SANTA CLARA,95054,2485 AUGUSTINE DRIVE,,US,DE,941692300.0,,,1-LAF,1,1231.0,10-K,20191231,2019.0,FY,20200204,2020-02-04 17:22:00.0,0,1,amdform10-kfy2019_htm.xml,1,,q1,2020',
 '0000002969-20-000010,2969,AIR PRODUCTS & CHEMICALS INC /DE/,2810.0,US,PA,ALLENTOWN,18195-1501,7201 HAMILTON BLVD,,6104814911,US,PA,AL

In [15]:
tag_rdd.take(5)

['tag,version,custom,abstract,datatype,iord,crdr,tlabel,doc,quarter,year',
 'OperatingLeasesRentExpenseNet,us-gaap/2018,0,0,monetary,D,D,"Operating Leases, Rent Expense, Net","Rental expense for the reporting period incurred under operating leases, including minimum and any contingent rent expense, net of related sublease income.",q1,2020',
 'OperatingLeaseVariableLeaseIncome,us-gaap/2018,0,0,monetary,D,C,"Operating Lease, Variable Lease Income","Amount of operating lease income from variable lease payments paid and payable to lessor, excluding amount included in measurement of lease receivable.",q1,2020',
 'OperatingLeaseWeightedAverageDiscountRatePercent,us-gaap/2018,0,0,percent,I,,"Operating Lease, Weighted Average Discount Rate, Percent",Weighted average discount rate for operating lease calculated at point in time.,q1,2020',
 'DeferredCompensationArrangementWithIndividualCompensationExpense,us-gaap/2018,0,0,monetary,D,D,"Deferred Compensation Arrangement with Individual, Compensat

### propressing: splitting

In [4]:
nums_split = nums_rdd.map(lambda x: x.split(","))
pre_split  = pre_rdd.map(lambda x: x.split(","))
sub_split  = sub_rdd.map(lambda x: x.split(","))
tag_split  = tag_rdd.map(lambda x: x.split(","))

In [17]:
nums_split.take(3)

[['adsh',
  'tag',
  'version',
  'ddate',
  'qtrs',
  'uom',
  'segments',
  'coreg',
  'value',
  'footnote',
  'quarter',
  'year'],
 ['0001564590-20-010652',
  'AccountsPayableCurrentAndNoncurrent',
  'us-gaap/2019',
  '20181231',
  '0',
  'USD',
  '',
  '',
  '607000.0',
  '',
  'q1',
  '2020'],
 ['0000753308-20-000021',
  'LongTermDebtCurrent',
  'us-gaap/2019',
  '20181231',
  '0',
  'USD',
  'LegalEntity=NexteraEnergyResources;',
  'NexteraEnergyResources',
  '602000000.0',
  '',
  'q1',
  '2020']]

### see nubmer of records in each table

In [ ]:
# per company * per quater * per element * per segement 
nums_split.count()

11493263

In [ ]:
# line of report
pre_split.count()

2746310

In [ ]:
# number of report
sub_split.count()

24940

In [ ]:
# number of financial program
tag_split.count()

298803

### change to no header

In [5]:
nums_header = nums_split.first()
nums_no_header = nums_split.filter(lambda row: row != nums_header)

In [23]:
nums_header

['adsh',
 'tag',
 'version',
 'ddate',
 'qtrs',
 'uom',
 'segments',
 'coreg',
 'value',
 'footnote',
 'quarter',
 'year']

#### apply no header to the rest tables

In [14]:
sub_header = sub_split.first()
sub_no_header = sub_split.filter(lambda row: row != sub_header)

tag_header = tag_split.first()
tag_no_header = tag_split.filter(lambda row: row != tag_header)

pre_header = pre_split.first()
pre_no_header = pre_split.filter(lambda row: row != pre_header)

### Count unique tags in tag_2020.csv

In [ ]:
# index of the 'tag' column
tag_col = tag_header.index("tag")  

tag_values = tag_no_header.map(lambda row: row[tag_col])
tag_counts = tag_values.countByValue()

list(tag_counts.items())[:25] # frequency table

[('OperatingLeasesRentExpenseNet', 10),
 ('OperatingLeaseVariableLeaseIncome', 6),
 ('OperatingLeaseWeightedAverageDiscountRatePercent', 4),
 ('DeferredCompensationArrangementWithIndividualCompensationExpense', 8),
 ('DeferredCompensationEquity', 9),
 ('DeferredCompensationLiabilityClassifiedNoncurrent', 9),
 ('DeferredCompensationLiabilityCurrent', 10),
 ('DeferredCompensationLiabilityCurrentAndNoncurrent', 7),
 ('OriginationOfLoansToEmployeeStockOwnershipPlans', 7),
 ('OriginationOfNotesReceivableFromRelatedParties', 9),
 ('OtherAccountsPayableAndAccruedLiabilities', 9),
 ('OtherAccruedLiabilitiesCurrent', 9),
 ('OtherAccruedLiabilitiesNoncurrent', 10),
 ('OtherAdditionalCapital', 9),
 ('OtherAmortizationOfDeferredCharges', 10),
 ('OtherAssetImpairmentCharges', 9),
 ('OtherAssets', 15),
 ('OtherAssetsCurrent', 10),
 ('OtherAssetsFairValueDisclosure', 7),
 ('OtherAssetsMiscellaneous', 8),
 ('DeferredCosts', 9),
 ('DeferredCostsAndOtherAssets', 8),
 ('DeferredCostsCurrent', 8),
 ('Defe

### Find the top 10 most frequent columns in nums

In [6]:
segments_col = nums_header.index("segments")
segments_values = nums_no_header.map(lambda row: row[segments_col])
result = segments_values.countByValue()

sorted(result.items(), key=lambda x: x[1], reverse=True)[:10]

[('', 5932914),
 ('EquityComponents=CommonStock;', 422184),
 ('EquityComponents=AdditionalPaidInCapital;', 332485),
 ('EquityComponents=RetainedEarnings;', 295699),
 ('EquityComponents=AccumulatedOtherComprehensiveIncome;', 173541),
 ('EquityComponents=TreasuryStock;', 104556),
 ('EquityComponents=NoncontrollingInterest;', 95139),
 ('ConsolidatedEntities=ParentCompany;', 90242),
 ('ConsolidationItems=ConsolidationEliminations;', 67656),
 ('EquityComponents=Parent;', 60877)]

In [7]:
#Assets, Revenue, NetIncome ...  ---> consolidated 
#EBITA 

###  Explore one important tag: assests, net income, revenue ...

### Total number of numeric facts in NUM

In [22]:
nums_split.map(lambda r: 1).reduce(lambda a,b: a+b)

11493263

In [31]:
# key: (tag, version)
num_k = nums_rdd.map(lambda r: ((r["tag"], r["version"]), float(r["value"])))
tag_k = tag_rdd.map(lambda r: ((r["tag"], r["version"]), r["datatype"]))

joined = num_k.join(tag_k)
joined


PythonRDD[111] at RDD at PythonRDD.scala:56

In [34]:
print(nums_split.first())
print(type(nums_rdd.first()))

print(tag_split.first())
print(type(tag_rdd.first()))


['adsh', 'tag', 'version', 'ddate', 'qtrs', 'uom', 'segments', 'coreg', 'value', 'footnote', 'quarter', 'year']
<class 'str'>
['tag', 'version', 'custom', 'abstract', 'datatype', 'iord', 'crdr', 'tlabel', 'doc', 'quarter', 'year']
<class 'str'>


In [35]:
# key: (tag, version)
num_k = nums_split.map(
    lambda cols: ((cols[1], cols[2]), float(cols[8]) if cols[8] not in ("", None) else 0.0)
)

tag_k = tag_split.map(
    lambda cols: ((cols[0], cols[1]), cols[4])   # datatype
)

joined = num_k.join(tag_k)


In [38]:
# rows with too few columns in NUM
bad_num_short = nums_split.filter(lambda cols: len(cols) < 9).take(10)
print("Bad NUM (short):", bad_num_short)

# rows with too few columns in TAG
bad_tag_short = tag_split.filter(lambda cols: len(cols) < 5).take(10)
print("Bad TAG (short):", bad_tag_short)


Bad NUM (short): []
Bad TAG (short): []


In [39]:
def make_num_pair(cols):
    # need at least tag, version, value
    if len(cols) <= 8:
        return None
    val_str = cols[8]
    try:
        v = float(val_str) if val_str not in ("", None) else 0.0
    except Exception:
        # malformed numeric string; skip this row
        return None
    return ((cols[1], cols[2]), v)   # ( (tag, version), value )

num_k = (nums_split
         .map(make_num_pair)
         .filter(lambda x: x is not None))


In [40]:
def make_tag_pair(cols):
    if len(cols) <= 4:
        return None
    return ((cols[0], cols[1]), cols[4])   # ( (tag, version), datatype )

tag_k = (tag_split
         .map(make_tag_pair)
         .filter(lambda x: x is not None))


In [44]:
def safe_float(s):
    try:
        if s is None or s == "":
            return None
        return float(s)
    except Exception:
        return None


In [ ]:
# value or 0
values = nums_split.map(lambda cols: safe_float(cols[8]) or 0.0)

total_sum = values.reduce(lambda a, b: a + b)
print("Total sum of all NUM values:", total_sum)


Total sum of all NUM values: 1.9483985761004333e+17


In [47]:
def seq_op(acc, cols):
    s, c = acc
    v = safe_float(cols[8])
    if v is None:
        return (s, c)
    return (s + v, c + 1)

def comb_op(acc1, acc2):
    return (acc1[0] + acc2[0], acc1[1] + acc2[1])

sum_count = nums_split.aggregate((0.0, 0), seq_op, comb_op)

total_sum, total_count = sum_count
avg_value = total_sum / total_count if total_count > 0 else float("nan")

print("Total sum:", total_sum)
print("Count of non-null values:", total_count)
print("Average value:", avg_value)


Total sum: 1.9483985761004333e+17
Count of non-null values: 11007145
Average value: 17701216583.414078


In [48]:
tag_values = (nums_split
              .map(lambda cols: (cols[1], safe_float(cols[8])))
              .filter(lambda kv: kv[1] is not None))   # 过滤掉无效值

tag_sum = tag_values.reduceByKey(lambda a, b: a + b)

print("Sample tag sums:", tag_sum.take(10))


Sample tag sums: [('TotalStockholdersEquityBeforyEsopAndTreasuryStock', 312933883.0), ('RepaymentsOfDebtAndCapitalLeaseObligations', 665257568099.0), ('PaymentsOfDividends', 4097650388990.0), ('DeferredTaxAssets', 89965198779983.0), ('CommonStockSharesOutstanding', 164806378707772.7), ('CashFlowsFromUsedInInvestingActivities', -409687099399890.0), ('Depositsandprepaidhealthinsurance', 374687000.0), ('InvestmentOwnedPercentOfNetAssets', 1243.943), ('MembersEquity', 5488533380342.0), ('SellingAndMarketingExpense', 2056454666885.0)]


In [ ]:
year_values = (nums_split
               .map(lambda cols: (cols[11], safe_float(cols[8])))
               .filter(lambda kv: kv[1] is not None))

year_sum_count = year_values.aggregateByKey(
    (0.0, 0),                              # (sum, count)
    lambda acc, v: (acc[0] + v, acc[1] + 1),   # In every partition 
    lambda a, b: (a[0] + b[0], a[1] + b[1])    # In between partition
)

year_avg = year_sum_count.mapValues(
    lambda sc: sc[0] / sc[1] if sc[1] > 0 else float("nan")
)

print("Yearly averages:", year_avg.take(10))


Yearly averages: [(' and a 211m decrease arising from the application of the new ECL impairment methodology', 127500000.0), (' Versamark in the Enterprise Inkjet Systems segment and Digimaster in the Print Systems segment."', 31000000.0), (' for fiscal 2019 and 2018', 60000000.0), (' respectively) in costs and expenses applicable to revenues', 60019000.0), ('680', 1.0), (' ""Income Statement - Reporting Comprehensive Income: Reclassification of Certain Tax Effects from Accumulated Other Comprehensive Income"" in the first quarter of 2019."', -7700000.0), (' the work on implementing the ring-fencing requirements and the integration of MBNA); and the fair value unwind and other items (loss of 270 million)."', 228000000.0), ('(1) Includes share-based compensation expense as follows:2019 2018 2019 2018Cost of sales$5.7 $3.4 $15.8 $10.9Research and development$21.2 $19.4 $63.0 $53.2Selling', 57000000.0), ('007 million (2018  $666 million)', 2699000000.0), ('772) for the three months ended D